# Lab Assignment 3 - Part A: Tabular Medical Data
## Pima Indians Diabetes Dataset

Pipeline: acquire -> inspect -> clean (implausible clinical values) -> impute
-> EDA -> hypothesis testing (t-test, chi-square, ANOVA) -> feature engineering
-> feature selection & extraction -> scaling -> model-ready tensors

Dataset: https://www.kaggle.com/datasets/jamaltariqcheema/pimaindians-diabetes-dataset
Place `diabetes.csv` in ./data/

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import SelectKBest, f_classif, mutual_info_classif, RFE
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
RANDOM_STATE = 42

DATA_PATH = Path("data/diabetes.csv")
OUT_DIR = Path("outputs/partA")
OUT_DIR.mkdir(parents=True, exist_ok=True)

## A1. Data acquisition and inspection

In [ ]:
df = pd.read_csv(DATA_PATH)

print("Shape:", df.shape)
print("\nColumns:", list(df.columns))
print("\nDtypes:\n", df.dtypes)
print("\nHead:\n", df.head())
print("\nDescribe:\n", df.describe().T)
print("\nNulls reported by pandas:\n", df.isna().sum())
print("\nDuplicated rows:", df.duplicated().sum())
print("\nClass balance:\n", df["Outcome"].value_counts(normalize=True))

## A2. Healthcare-specific cleaning: implausible clinical values

This dataset reports missing measurements as `0`, which is a *valid-looking*
number. A generic `isna()` check finds nothing. But a Glucose of 0 or a BMI of 0
is biologically impossible in a living patient, so these are disguised nulls.
Failing to catch this is the classic healthcare-data trap: the model silently
learns from impossible physiology.

Pregnancies and Outcome are legitimately 0, so they are excluded.

In [ ]:
IMPLAUSIBLE_ZERO_COLS = [
    "Glucose",
    "BloodPressure",
    "SkinThickness",
    "Insulin",
    "BMI",
]

# Clinically plausible ranges used as a second sanity filter (min, max)
PLAUSIBLE_RANGES = {
    "Glucose": (40, 400),          # mg/dL
    "BloodPressure": (30, 200),    # mmHg diastolic
    "SkinThickness": (5, 100),     # mm triceps fold
    "Insulin": (10, 900),          # mu U/ml
    "BMI": (10, 80),               # kg/m^2
    "Age": (18, 120),              # years
    "DiabetesPedigreeFunction": (0.0, 3.0),
}

df_clean = df.copy()

print("Zero counts before treatment:")
for col in IMPLAUSIBLE_ZERO_COLS:
    n_zero = (df_clean[col] == 0).sum()
    print(f"  {col:<16} {n_zero:>4} zeros ({n_zero / len(df_clean):.1%})")

# Step 1: recode implausible zeros as true missing values
df_clean[IMPLAUSIBLE_ZERO_COLS] = df_clean[IMPLAUSIBLE_ZERO_COLS].replace(0, np.nan)

# Step 2: clip anything outside a plausible physiological range to NaN as well
for col, (lo, hi) in PLAUSIBLE_RANGES.items():
    if col in df_clean.columns:
        mask = (df_clean[col] < lo) | (df_clean[col] > hi)
        if mask.sum():
            print(f"  {col}: {mask.sum()} value(s) outside [{lo}, {hi}] -> NaN")
        df_clean.loc[mask, col] = np.nan

print("\nTrue missingness after recoding:\n", df_clean.isna().sum())

In [ ]:
# Missingness pattern - is data missing at random, or structurally?
fig, ax = plt.subplots(figsize=(10, 4))
sns.heatmap(df_clean.isna().T, cbar=False, cmap="viridis", ax=ax)
ax.set_title("Missingness pattern (yellow = missing)")
plt.tight_layout()
plt.savefig(OUT_DIR / "missingness_pattern.png", dpi=120)
plt.close()

# Does missingness itself correlate with the outcome? (informative missingness)
print("\nMissingness rate by Outcome class:")
print(df_clean.groupby(df["Outcome"])[IMPLAUSIBLE_ZERO_COLS].apply(
    lambda g: g.isna().mean()
))

# Retain missingness as an explicit feature - in clinical data, the fact that a
# test was NOT ordered is itself signal (a clinician did not think it necessary).
df_clean["Insulin_missing"] = df_clean["Insulin"].isna().astype(int)
df_clean["SkinThickness_missing"] = df_clean["SkinThickness"].isna().astype(int)

## A3. Train/test split BEFORE imputation and scaling

Imputing on the full dataset leaks test-set statistics into training. The split
comes first; every transformer is fitted on train only.

In [ ]:
X = df_clean.drop(columns=["Outcome"])
y = df_clean["Outcome"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
print("Train:", X_train.shape, " Test:", X_test.shape)

# Median imputation - robust to the heavy right skew in Insulin and SkinThickness
imputer = SimpleImputer(strategy="median")
X_train_imp = pd.DataFrame(
    imputer.fit_transform(X_train), columns=X_train.columns, index=X_train.index
)
X_test_imp = pd.DataFrame(
    imputer.transform(X_test), columns=X_test.columns, index=X_test.index
)
print("\nRemaining NaNs after imputation:", X_train_imp.isna().sum().sum())

# Full imputed frame for EDA and statistical testing (train only, to stay honest)
eda = X_train_imp.copy()
eda["Outcome"] = y_train.values

## A4. Exploratory Data Analysis

In [ ]:
# Distributions by class
fig, axes = plt.subplots(3, 3, figsize=(15, 11))
for ax, col in zip(axes.ravel(), X.columns):
    sns.kdeplot(data=eda, x=col, hue="Outcome", fill=True, common_norm=False, ax=ax)
    ax.set_title(col)
for ax in axes.ravel()[len(X.columns):]:
    ax.axis("off")
plt.suptitle("Feature distributions by diabetes outcome", y=1.00)
plt.tight_layout()
plt.savefig(OUT_DIR / "distributions_by_class.png", dpi=120)
plt.close()

# Correlation structure
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(eda.corr(), annot=True, fmt=".2f", cmap="RdBu_r", center=0, ax=ax)
ax.set_title("Correlation matrix")
plt.tight_layout()
plt.savefig(OUT_DIR / "correlation_matrix.png", dpi=120)
plt.close()

# Outliers
fig, ax = plt.subplots(figsize=(12, 5))
sns.boxplot(data=eda[X.columns].apply(stats.zscore), orient="h", ax=ax)
ax.set_title("Standardised feature spread (outlier check)")
plt.tight_layout()
plt.savefig(OUT_DIR / "outlier_boxplot.png", dpi=120)
plt.close()

print("Class imbalance (train):")
print(y_train.value_counts(normalize=True).rename("proportion"))

## A5. Hypothesis test - independent two-sample t-test

H0: mean plasma glucose is equal in diabetic and non-diabetic patients.
H1: the means differ.

Welch's t-test is used because the two groups have unequal variance.
Normality is checked first with Shapiro-Wilk; if violated, the non-parametric
Mann-Whitney U test is reported alongside.

In [ ]:
def two_group_test(frame, feature, group_col="Outcome", alpha=0.05):
    g0 = frame.loc[frame[group_col] == 0, feature].dropna()
    g1 = frame.loc[frame[group_col] == 1, feature].dropna()

    # Assumption checks
    _, p_norm0 = stats.shapiro(g0.sample(min(len(g0), 500), random_state=RANDOM_STATE))
    _, p_norm1 = stats.shapiro(g1.sample(min(len(g1), 500), random_state=RANDOM_STATE))
    _, p_var = stats.levene(g0, g1)

    t_stat, p_t = stats.ttest_ind(g0, g1, equal_var=False)   # Welch
    u_stat, p_u = stats.mannwhitneyu(g0, g1, alternative="two-sided")

    # Cohen's d effect size
    pooled_sd = np.sqrt(((len(g0) - 1) * g0.var() + (len(g1) - 1) * g1.var())
                        / (len(g0) + len(g1) - 2))
    cohens_d = (g1.mean() - g0.mean()) / pooled_sd

    print(f"\n--- {feature} ---")
    print(f"  non-diabetic: n={len(g0)}, mean={g0.mean():.2f}, sd={g0.std():.2f}")
    print(f"  diabetic:     n={len(g1)}, mean={g1.mean():.2f}, sd={g1.std():.2f}")
    print(f"  Shapiro p (grp0/grp1): {p_norm0:.4g} / {p_norm1:.4g}"
          f"  -> {'non-normal' if min(p_norm0, p_norm1) < alpha else 'normal'}")
    print(f"  Levene p: {p_var:.4g}"
          f"  -> {'unequal' if p_var < alpha else 'equal'} variances")
    print(f"  Welch t = {t_stat:.3f}, p = {p_t:.4g}")
    print(f"  Mann-Whitney U = {u_stat:.0f}, p = {p_u:.4g}")
    print(f"  Cohen's d = {cohens_d:.3f}")
    print(f"  Decision: {'REJECT H0' if p_t < alpha else 'FAIL TO REJECT H0'} at alpha={alpha}")
    return {"feature": feature, "t": t_stat, "p_ttest": p_t,
            "p_mannwhitney": p_u, "cohens_d": cohens_d}


ttest_results = [two_group_test(eda, f) for f in
                 ["Glucose", "BMI", "Age", "Insulin", "BloodPressure"]]
ttest_df = pd.DataFrame(ttest_results)

# Multiple-comparison correction: five simultaneous tests inflate the false
# positive rate, so Bonferroni-adjust the alpha.
alpha_bonf = 0.05 / len(ttest_df)
ttest_df["significant_bonferroni"] = ttest_df["p_ttest"] < alpha_bonf
print(f"\nBonferroni-corrected alpha = {alpha_bonf:.4f}")
print(ttest_df[["feature", "p_ttest", "cohens_d", "significant_bonferroni"]])

## A6. Chi-square test of independence

H0: BMI category and diabetes outcome are independent.
H1: they are associated.

Chi-square needs categorical variables, so continuous BMI is binned into the
standard WHO clinical categories.

In [ ]:
bmi_bins = [0, 18.5, 25, 30, 35, 100]
bmi_labels = ["Underweight", "Normal", "Overweight", "Obese I", "Obese II+"]
eda["BMI_category"] = pd.cut(eda["BMI"], bins=bmi_bins, labels=bmi_labels)

age_bins = [20, 30, 40, 50, 100]
age_labels = ["21-30", "31-40", "41-50", "51+"]
eda["Age_group"] = pd.cut(eda["Age"], bins=age_bins, labels=age_labels)


def chi_square_test(frame, cat_col, target="Outcome", alpha=0.05):
    contingency = pd.crosstab(frame[cat_col], frame[target])
    chi2, p, dof, expected = stats.chi2_contingency(contingency)

    # Cramer's V effect size
    n = contingency.values.sum()
    cramers_v = np.sqrt(chi2 / (n * (min(contingency.shape) - 1)))

    print(f"\n--- Chi-square: {cat_col} vs {target} ---")
    print("Observed:\n", contingency)
    print("\nExpected:\n", pd.DataFrame(expected, index=contingency.index,
                                        columns=contingency.columns).round(1))
    low_expected = (expected < 5).sum()
    if low_expected:
        print(f"  WARNING: {low_expected} cell(s) with expected count < 5 "
              f"-> chi-square approximation unreliable, consider Fisher's exact")
    print(f"  chi2 = {chi2:.3f}, dof = {dof}, p = {p:.4g}")
    print(f"  Cramer's V = {cramers_v:.3f}")
    print(f"  Decision: {'REJECT H0' if p < alpha else 'FAIL TO REJECT H0'} "
          f"-> variables are {'associated' if p < alpha else 'independent'}")
    return {"variable": cat_col, "chi2": chi2, "p": p, "cramers_v": cramers_v}


chi_results = [chi_square_test(eda, c) for c in ["BMI_category", "Age_group"]]
chi_df = pd.DataFrame(chi_results)

## A7. One-way ANOVA

H0: mean glucose is equal across all four age groups.
H1: at least one age group differs.

ANOVA assumes normality within groups and homogeneity of variance. Both are
checked; Kruskal-Wallis is reported as the non-parametric backup. A significant
F-test only says "somewhere there is a difference", so Tukey HSD identifies
which specific pairs differ.

In [ ]:
def one_way_anova(frame, value_col, group_col, alpha=0.05):
    groups = [g[value_col].dropna().values
              for _, g in frame.groupby(group_col, observed=True)]
    names = [str(k) for k, _ in frame.groupby(group_col, observed=True)]

    _, p_levene = stats.levene(*groups)
    f_stat, p_anova = stats.f_oneway(*groups)
    h_stat, p_kruskal = stats.kruskal(*groups)

    # Eta-squared effect size
    grand_mean = frame[value_col].mean()
    ss_between = sum(len(g) * (g.mean() - grand_mean) ** 2 for g in groups)
    ss_total = ((frame[value_col] - grand_mean) ** 2).sum()
    eta_sq = ss_between / ss_total

    print(f"\n--- One-way ANOVA: {value_col} across {group_col} ---")
    for n, g in zip(names, groups):
        print(f"  {n:<10} n={len(g):<4} mean={g.mean():7.2f}  sd={g.std():6.2f}")
    print(f"  Levene p = {p_levene:.4g} -> "
          f"{'unequal' if p_levene < alpha else 'equal'} variances")
    print(f"  F = {f_stat:.3f}, p = {p_anova:.4g}")
    print(f"  Kruskal-Wallis H = {h_stat:.3f}, p = {p_kruskal:.4g}")
    print(f"  eta^2 = {eta_sq:.3f}")
    print(f"  Decision: {'REJECT H0' if p_anova < alpha else 'FAIL TO REJECT H0'}")
    return f_stat, p_anova


one_way_anova(eda, "Glucose", "Age_group")
one_way_anova(eda, "Glucose", "BMI_category")

# Post-hoc pairwise comparison
try:
    from statsmodels.stats.multicomp import pairwise_tukeyhsd
    tukey = pairwise_tukeyhsd(
        endog=eda["Glucose"],
        groups=eda["Age_group"].astype(str),
        alpha=0.05,
    )
    print("\nTukey HSD post-hoc (Glucose ~ Age_group):")
    print(tukey)
except ImportError:
    print("\n[statsmodels not installed - skipping Tukey HSD]")
    print("    pip install statsmodels")

## A8. Feature engineering

Domain-driven features, not blind polynomial expansion. Each one encodes a
known clinical relationship.

In [ ]:
def engineer_features(frame: pd.DataFrame) -> pd.DataFrame:
    out = frame.copy()

    # Clinical categories (ordinal encoding preserves the natural ordering)
    out["BMI_cat"] = pd.cut(out["BMI"], bins=bmi_bins, labels=False)
    out["Age_bin"] = pd.cut(out["Age"], bins=age_bins, labels=False)

    # Glucose-to-insulin ratio: a crude proxy for insulin resistance
    out["Glucose_Insulin_ratio"] = out["Glucose"] / out["Insulin"].replace(0, np.nan)

    # HOMA-IR-like index (simplified surrogate for insulin resistance)
    out["HOMA_IR_proxy"] = (out["Glucose"] * out["Insulin"]) / 405.0

    # Interaction terms with known clinical basis
    out["BMI_x_Age"] = out["BMI"] * out["Age"]
    out["Glucose_x_BMI"] = out["Glucose"] * out["BMI"]

    # Genetic risk amplified by age
    out["Pedigree_x_Age"] = out["DiabetesPedigreeFunction"] * out["Age"]

    # Binary clinical flags mirroring diagnostic thresholds
    out["is_hyperglycemic"] = (out["Glucose"] >= 140).astype(int)
    out["is_obese"] = (out["BMI"] >= 30).astype(int)
    out["is_hypertensive"] = (out["BloodPressure"] >= 90).astype(int)

    # Log-transform the heavily right-skewed features
    for col in ["Insulin", "DiabetesPedigreeFunction", "Age"]:
        out[f"log_{col}"] = np.log1p(out[col])

    return out.fillna(out.median(numeric_only=True))


X_train_fe = engineer_features(X_train_imp)
X_test_fe = engineer_features(X_test_imp)
print("Features after engineering:", X_train_fe.shape[1])
print(list(X_train_fe.columns))

## A9. Feature selection and extraction

In [ ]:
# --- Filter method: ANOVA F-score ---
selector_f = SelectKBest(score_func=f_classif, k=10).fit(X_train_fe, y_train)
f_scores = pd.Series(selector_f.scores_, index=X_train_fe.columns).sort_values(ascending=False)
print("\nTop 10 by ANOVA F-score:\n", f_scores.head(10).round(2))

# --- Filter method: mutual information (captures non-linear dependence) ---
mi = pd.Series(
    mutual_info_classif(X_train_fe, y_train, random_state=RANDOM_STATE),
    index=X_train_fe.columns,
).sort_values(ascending=False)
print("\nTop 10 by mutual information:\n", mi.head(10).round(4))

# --- Wrapper method: recursive feature elimination ---
rfe = RFE(
    LogisticRegression(max_iter=2000, random_state=RANDOM_STATE),
    n_features_to_select=10,
).fit(StandardScaler().fit_transform(X_train_fe), y_train)
rfe_selected = X_train_fe.columns[rfe.support_].tolist()
print("\nRFE selected:", rfe_selected)

# --- Embedded method: random forest importance ---
rf = RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE).fit(X_train_fe, y_train)
rf_imp = pd.Series(rf.feature_importances_, index=X_train_fe.columns).sort_values(ascending=False)
print("\nTop 10 by random forest importance:\n", rf_imp.head(10).round(4))

fig, ax = plt.subplots(figsize=(9, 7))
rf_imp.head(15).sort_values().plot.barh(ax=ax)
ax.set_title("Random forest feature importance (top 15)")
plt.tight_layout()
plt.savefig(OUT_DIR / "feature_importance.png", dpi=120)
plt.close()

# Consensus: features endorsed by at least two of the four methods
votes = pd.Series(0, index=X_train_fe.columns)
for chosen in [f_scores.head(10).index, mi.head(10).index,
               pd.Index(rfe_selected), rf_imp.head(10).index]:
    votes[chosen] += 1
selected_features = votes[votes >= 2].index.tolist()
print(f"\nConsensus feature set ({len(selected_features)}):", selected_features)

In [ ]:
# --- Feature extraction: PCA ---
scaler_pca = StandardScaler().fit(X_train_fe)
pca_full = PCA(random_state=RANDOM_STATE).fit(scaler_pca.transform(X_train_fe))
cumvar = np.cumsum(pca_full.explained_variance_ratio_)
n_components = int(np.argmax(cumvar >= 0.95) + 1)
print(f"\nComponents needed for 95% variance: {n_components}")

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(range(1, len(cumvar) + 1), cumvar, marker="o")
ax.axhline(0.95, ls="--", c="red", label="95% variance")
ax.axvline(n_components, ls="--", c="green")
ax.set_xlabel("Number of components")
ax.set_ylabel("Cumulative explained variance")
ax.set_title("PCA scree plot")
ax.legend()
plt.tight_layout()
plt.savefig(OUT_DIR / "pca_scree.png", dpi=120)
plt.close()

pca = PCA(n_components=n_components, random_state=RANDOM_STATE)
X_train_pca = pca.fit_transform(scaler_pca.transform(X_train_fe))
X_test_pca = pca.transform(scaler_pca.transform(X_test_fe))
print("PCA-transformed shape:", X_train_pca.shape)

## A10. Scaling to model-ready tensors

StandardScaler is fitted on training data only, then applied to test.

In [ ]:
scaler = StandardScaler().fit(X_train_fe[selected_features])
X_train_scaled = scaler.transform(X_train_fe[selected_features])
X_test_scaled = scaler.transform(X_test_fe[selected_features])

print("Train tensor:", X_train_scaled.shape, "dtype:", X_train_scaled.dtype)
print("Test tensor: ", X_test_scaled.shape)
print("Train mean ~0:", np.round(X_train_scaled.mean(axis=0)[:5], 6))
print("Train std  ~1:", np.round(X_train_scaled.std(axis=0)[:5], 6))

# Class imbalance handling - compute weights rather than resampling, so no
# synthetic patients are invented.
classes, counts = np.unique(y_train, return_counts=True)
class_weights = {int(c): len(y_train) / (len(classes) * n)
                 for c, n in zip(classes, counts)}
print("\nClass weights for training:", {k: round(v, 3) for k, v in class_weights.items()})

np.savez_compressed(
    OUT_DIR / "pima_model_ready.npz",
    X_train=X_train_scaled, X_test=X_test_scaled,
    y_train=y_train.values, y_test=y_test.values,
    X_train_pca=X_train_pca, X_test_pca=X_test_pca,
    feature_names=np.array(selected_features),
)
print(f"\nSaved model-ready arrays to {OUT_DIR / 'pima_model_ready.npz'}")

## Part A summary

| Step | Finding |
|---|---|
| Cleaning | 5 columns encoded missingness as `0`; Insulin ~49% and SkinThickness ~30% missing |
| t-test | Glucose differs strongly between classes (large effect); BloodPressure weakest |
| Chi-square | BMI category and Age group both associated with outcome |
| ANOVA | Mean glucose differs significantly across age groups |
| Features | 8 original -> ~20 engineered -> ~10 by 4-method consensus |
| Output | Scaled train/test tensors + PCA variant + class weights |